# Visualization - Bengaluru House Price Data
Notebook trực quan hóa dữ liệu bất động sản Bengaluru

In [ ]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- HÌNH 1: Tải và khám phá dữ liệu gốc ---
df1 = pd.read_csv('Bengaluru_House_Data.csv')
print('Shape:', df1.shape)
print('\nCác cột:', df1.columns.tolist())
print('\nSố lượng giá trị null:')
print(df1.isnull().sum())
df1.head()

In [ ]:
# --- Đổi tên cột cho giống với paper ---
df2 = df1.rename(columns={'total_sqft': 'Area', 'size': 'Room', 'bath': 'Bathroom', 'price': 'Price', 'location': 'Location'})

# --- HÌNH 2: Cài đặt giá trị còn thiếu (Setting Missing Values) ---
median = df2['Bathroom'].median()
df2['Bathroom'] = df2['Bathroom'].fillna(median)
print(f'Median Bathroom: {median}')
print(f'Null còn lại sau fillna:\n{df2.isnull().sum()}')

In [ ]:
# --- Bỏ các hàng null còn lại và xử lý dữ liệu ---
df3 = df2.dropna()
print(f'Shape sau dropna: {df3.shape}')

# --- HÌNH 3: Tạo đặc trưng Bhk từ Room ---
df5 = df3.copy()
df5['Bhk'] = df5['Room'].apply(lambda x: int(x.split(' ')[0]) if isinstance(x, str) and ' ' in x else None)
print(f'Bhk unique: {sorted(df5["Bhk"].dropna().unique())}')

# Lọc và chuyển đổi cột Area sang float (bỏ dạng range như '1133 - 1384')
def is_float(x):
    try:
        float(x)
        return True
    except:
        return False

df5 = df5[df5['Area'].apply(is_float)].copy()
df5['Area'] = df5['Area'].astype(float)
print(f'Shape sau lọc Area: {df5.shape}')
print(f'Area dtype: {df5["Area"].dtype}')

In [ ]:
# --- HÌNH 4: Giảm chiều dữ liệu Location ---
df5['Location'] = df5['Location'].apply(lambda x: x.strip() if isinstance(x, str) else x)
location_stats = df5.groupby('Location')['Location'].agg('count').sort_values(ascending=False)
location_stats_less_than_10 = location_stats[location_stats <= 10]

df6 = df5.copy()
df6['Location'] = df6['Location'].apply(lambda x: 'other' if x in location_stats_less_than_10 else x)
print(f'Unique Locations trước: {len(df5.Location.unique())}')
print(f'Unique Locations sau:  {len(df6.Location.unique())}')

# Đảm bảo Area là float trước khi tính toán
df6['Area'] = pd.to_numeric(df6['Area'], errors='coerce')
df6['Price'] = pd.to_numeric(df6['Price'], errors='coerce')
df6 = df6.dropna(subset=['Area', 'Price'])

# Tạo cột Price_per_sqft
df6['Price_per_sqft'] = df6['Price'] * 100000 / df6['Area']
print(f'\nShape cuối: {df6.shape}')
print(f'Area dtype: {df6["Area"].dtype}, Price dtype: {df6["Price"].dtype}')
df6.head()

In [ ]:
# --- HÌNH 5: Correlation Heatmap ---
plt.figure(figsize=(10, 7))
sns.heatmap(df6[['Area', 'Bathroom', 'Price', 'Bhk', 'Price_per_sqft']].corr(), 
            annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- HÌNH 6: Hàm vẽ Scatter Plot ---
def plot_scatter_chart(df, location):
    bhk2 = df[(df.Location == location) & (df.Bhk == 2)]
    bhk3 = df[(df.Location == location) & (df.Bhk == 3)]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(bhk2.Area, bhk2.Price, color='blue', label='2 BHK', s=50, alpha=0.6)
    ax.scatter(bhk3.Area, bhk3.Price, color='green', label='3 BHK', marker='+', s=50, alpha=0.6)
    ax.set_xlabel('Total Square Feet Area', fontsize=12)
    ax.set_ylabel('Price (Lakh Indian Rupees)', fontsize=12)
    ax.set_title(f'{location} - 2 BHK vs 3 BHK', fontsize=14)
    ax.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# =====================================================================
# HÌNH 7: Price BEFORE Outliers Removed - Rajaji Nagar
# Dữ liệu df6 chưa qua bước loại bỏ ngoại lai (trước outlier removal)
# =====================================================================
location = 'Rajaji Nagar'
df_loc = df6[df6.Location == location]

print(f'=== {location} - TRƯỚC KHI loại bỏ ngoại lai ===')
print(f'Số lượng dữ liệu: {len(df_loc)}')
print(f'Price min: {df_loc.Price.min():.2f}, max: {df_loc.Price.max():.2f}, mean: {df_loc.Price.mean():.2f}')
print(f'Price_per_sqft min: {df_loc.Price_per_sqft.min():.0f}, max: {df_loc.Price_per_sqft.max():.0f}')

# Scatter Plot BEFORE outliers removed
plot_scatter_chart(df6, location)

# Box Plot giá trước khi loại ngoại lai
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df_loc['Price'].dropna(), vert=True)
axes[0].set_title(f'{location} - Price (Before Outlier Removal)', fontsize=12)
axes[0].set_ylabel('Price (Lakh)')

axes[1].boxplot(df_loc['Price_per_sqft'].dropna(), vert=True)
axes[1].set_title(f'{location} - Price per Sqft (Before Outlier Removal)', fontsize=12)
axes[1].set_ylabel('Price per Sqft (Rupees)')

plt.tight_layout()
plt.show()

# Histogram giá trước khi loại ngoại lai
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_loc['Price'], bins=20, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].set_title(f'{location} - Phân phối Price (Before)', fontsize=12)
axes[0].set_xlabel('Price (Lakh)')
axes[0].set_ylabel('Count')

axes[1].hist(df_loc['Price_per_sqft'], bins=20, color='#e67e22', edgecolor='white', alpha=0.8)
axes[1].set_title(f'{location} - Phân phối Price/Sqft (Before)', fontsize=12)
axes[1].set_xlabel('Price per Sqft (Rupees)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# HÌNH 8: Loại bỏ ngoại lai và so sánh BEFORE vs AFTER
# =====================================================================

# Bước 1: Loại ngoại lai Area/Bhk (< 300 sqft/phòng)
df7 = df6[~(df6.Area / df6.Bhk < 300)]

# Bước 2: Loại ngoại lai Price_per_sqft theo mean +/- std từng location
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('Location'):
        m = np.mean(subdf.Price_per_sqft)
        st = np.std(subdf.Price_per_sqft)
        reduced_df = subdf[(subdf.Price_per_sqft > (m - st)) & (subdf.Price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df8 = remove_pps_outliers(df7)

# Bước 3: Loại ngoại lai Bathroom
df_clean = df8[df8.Bathroom < df8.Bhk + 2]

print(f'Trước loại ngoại lai: {len(df6)} mẫu')
print(f'Sau loại ngoại lai:   {len(df_clean)} mẫu')
print(f'Đã loại bỏ: {len(df6) - len(df_clean)} mẫu ({(len(df6) - len(df_clean)) / len(df6) * 100:.1f}%)')

In [ ]:
# =====================================================================
# HÌNH 9: So sánh BEFORE vs AFTER Outlier Removal - Rajaji Nagar
# =====================================================================
location = 'Rajaji Nagar'
df_before = df6[df6.Location == location]
df_after = df_clean[df_clean.Location == location]

print(f'=== {location} ===')
print(f'BEFORE: {len(df_before)} mẫu | Price mean: {df_before.Price.mean():.2f} Lakh')
print(f'AFTER:  {len(df_after)} mẫu  | Price mean: {df_after.Price.mean():.2f} Lakh')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# BEFORE
bhk2_b = df_before[df_before.Bhk == 2]
bhk3_b = df_before[df_before.Bhk == 3]
axes[0].scatter(bhk2_b.Area, bhk2_b.Price, color='#e74c3c', label='2 BHK', s=50, alpha=0.6)
axes[0].scatter(bhk3_b.Area, bhk3_b.Price, color='#e67e22', label='3 BHK', marker='+', s=60, alpha=0.7)
axes[0].set_xlabel('Area (sqft)', fontsize=12)
axes[0].set_ylabel('Price (Lakh)', fontsize=12)
axes[0].set_title(f'{location} - BEFORE Outlier Removal\n({len(df_before)} data points)', fontsize=13, color='#e74c3c')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AFTER
bhk2_a = df_after[df_after.Bhk == 2]
bhk3_a = df_after[df_after.Bhk == 3]
axes[1].scatter(bhk2_a.Area, bhk2_a.Price, color='#2ecc71', label='2 BHK', s=50, alpha=0.6)
axes[1].scatter(bhk3_a.Area, bhk3_a.Price, color='#3498db', label='3 BHK', marker='+', s=60, alpha=0.7)
axes[1].set_xlabel('Area (sqft)', fontsize=12)
axes[1].set_ylabel('Price (Lakh)', fontsize=12)
axes[1].set_title(f'{location} - AFTER Outlier Removal\n({len(df_after)} data points)', fontsize=13, color='#2ecc71')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'So sánh giá nhà {location}: Trước vs Sau loại ngoại lai', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Scatter Plot cho khu vực Hebbal ---
plot_scatter_chart(df6, 'Hebbal')

In [ ]:
# --- HÌNH 10: Phân phối giá theo Location (Top 10) ---
top_locations = df6.Location.value_counts().head(10).index
df_top = df6[df6.Location.isin(top_locations)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_top, x='Location', y='Price')
plt.xticks(rotation=45, ha='right')
plt.title('Phân phối giá nhà theo Top 10 Location', fontsize=14)
plt.ylabel('Price (Lakh)')
plt.tight_layout()
plt.show()

In [ ]:
# --- HÌNH 11: Histogram giá nhà ---
plt.figure(figsize=(10, 6))
plt.hist(df6.Price, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Price (Lakh Indian Rupees)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Phân phối giá nhà Bengaluru', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- HÌNH 12: Biểu đồ Price per sqft theo BHK ---
plt.figure(figsize=(10, 6))
sns.boxplot(data=df6, x='Bhk', y='Price_per_sqft')
plt.title('Price per Sqft theo số phòng ngủ (BHK)', fontsize=14)
plt.xlabel('BHK')
plt.ylabel('Price per Sqft (Rupees)')
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# HÌNH 13: Actual vs Predicted House Prices with Regression Lines
# =====================================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Tải dữ liệu đã xử lý
df_m = pd.read_csv('cleaned_data_for_model.csv')
cols_drop = [c for c in ['area_type', 'availability', 'society', 'balcony'] if c in df_m.columns]
if cols_drop:
    df_m.drop(cols_drop, axis='columns', inplace=True)

dummies = pd.get_dummies(df_m.Location)
df_dum = pd.concat([df_m, dummies.drop('other', axis='columns')], axis='columns')
df_model = df_dum.drop('Location', axis='columns')

X = df_model.drop('Price', axis='columns')
y = df_model.Price
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

# Train tất cả các mô hình
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1, max_iter=10000),
    'Decision Tree': DecisionTreeRegressor(random_state=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=10),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print(f'{name}: R2 = {r2_score(y_test, predictions[name]):.4f}')

In [ ]:
# --- Vẽ Actual vs Predicted cho tất cả mô hình ---
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#e67e22']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes_flat[idx]
    r2 = r2_score(y_test, y_pred)
    color = colors[idx]
    
    # Scatter: Actual vs Predicted
    ax.scatter(y_test, y_pred, alpha=0.4, s=20, color=color, edgecolors='white', linewidth=0.3)
    
    # Đường lý tưởng y = x (Perfect Prediction)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Prediction')
    
    # Regression line (best fit)
    z = np.polyfit(y_test, y_pred, 1)
    p = np.poly1d(z)
    x_line = np.linspace(min_val, max_val, 100)
    ax.plot(x_line, p(x_line), color=color, linewidth=2, label=f'Regression (slope={z[0]:.2f})')
    
    ax.set_xlabel('Actual Price (Lakh)', fontsize=11)
    ax.set_ylabel('Predicted Price (Lakh)', fontsize=11)
    ax.set_title(f'{name}\nR2 = {r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.2)

# Ẩn ô thừa (2x3 = 6 ô, chỉ dùng 5)
axes_flat[5].axis('off')

# Thêm bảng so sánh R2 vào ô trống
cell_text = [[name, f'{r2_score(y_test, yp):.4f}'] for name, yp in predictions.items()]
table = axes_flat[5].table(cellText=cell_text, colLabels=['Model', 'R2 Score'],
                           loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
# Tô màu header
for j in range(2):
    table[0, j].set_facecolor('#34495e')
    table[0, j].set_text_props(color='white', fontweight='bold')
# Highlight dòng R2 cao nhất
best_idx = max(range(len(cell_text)), key=lambda i: float(cell_text[i][1]))
for j in range(2):
    table[best_idx + 1, j].set_facecolor('#d5f5e3')
axes_flat[5].set_title('Model Comparison', fontsize=12, fontweight='bold')

plt.suptitle('Actual vs Predicted House Prices with Regression Lines', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()